# GNN Training and Serving Pipeline Test

This notebook demonstrates how to execute the GNN training pipeline and serving inference code entirely within a local notebook environment, hitting your actual Google Spanner database for data without deploying to Vertex AI or storing artifacts in Google Cloud Storage.

It uses the exact same code blocks found in:
- `gnn/src/pipeline/run_pipeline_local.py`
- `gnn/src/train_hetgnn.py`
- `gnn/src/serve.py`

In [ ]:
import os
import sys
import asyncio
import argparse
import datetime
from pathlib import Path
from unittest.mock import patch

# Ensure the gnn/src directory is in the Python path so we can import modules
src_path = str((Path.cwd().resolve().parent / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import train_hetgnn
import serve
import run_pipeline_local

## Configuration

Define the parameters for Spanner connectivity and the time windows for training and test data.

- **Training data**: `TRAIN_FROM` → `TRAIN_TO` — snapshots used to fit the model.
- **Test data**: `TEST_FROM` → `TEST_TO` — held-out snapshots used to evaluate inference.

In [ ]:
# Credential auth: check for networkagent.json in ../src (local dev), fall back to ADC.
# Mirrors the approach used in gnn/src/utils/data.py (SpannerDataset.__init__).
_local_creds = str((Path.cwd().resolve().parent / "src" / "networkagent.json").resolve())
creds_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS", _local_creds)
if creds_path and os.path.exists(creds_path):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = creds_path
    print(f"Using service-account credentials: {creds_path}")
else:
    print("networkagent.json not found — will use Application Default Credentials (ADC)")

project_id = os.getenv("GOOGLE_CLOUD_PROJECT", "agents-1234")
spanner_instance = os.getenv("SPANNER_INSTANCE", "networktopology-instance")
spanner_database = os.getenv("SPANNER_DATABASE", "networktopology-db")
interval_minutes = 5  # rca.md: 5-minute snapshot cadence
num_snapshots = 20  # Fallback used only when no time range is provided

# ── Time windows ──────────────────────────────────────────────────────────────
# Training data: April 14 18:00 → 19:45  (21 snapshots at 5-min intervals)
TRAIN_FROM = datetime.datetime(2026, 4, 14, 18,  0, 0)
TRAIN_TO   = datetime.datetime(2026, 4, 14, 19, 45, 0)

# Test / held-out data: April 14 19:45 → 20:00  (4 snapshots at 5-min intervals)
TEST_FROM  = datetime.datetime(2026, 4, 14, 19, 45, 0)
TEST_TO    = datetime.datetime(2026, 4, 14, 20,  0, 0)

# ── Args namespaces (shared Spanner config + per-window time bounds) ──────────
def _base_args(**kwargs):
    return argparse.Namespace(
        project=project_id,
        spanner_instance=spanner_instance,
        spanner_database=spanner_database,
        interval_minutes=interval_minutes,
        num_snapshots=num_snapshots,
        **kwargs,
    )

train_args = _base_args(from_time=TRAIN_FROM, to_time=TRAIN_TO)
test_args  = _base_args(from_time=TEST_FROM,  to_time=TEST_TO)

print(f"Training window : {TRAIN_FROM.isoformat()} → {TRAIN_TO.isoformat()}")
print(f"Test window     : {TEST_FROM.isoformat()}  → {TEST_TO.isoformat()}")

output_dir = Path("notebook_artifacts")
output_dir.mkdir(exist_ok=True)
snapshots_dir      = output_dir / "snapshots"
test_snapshots_dir = output_dir / "test_snapshots"

## 1. Ingest Training Data

Fetch network topology and metrics for the **training window** (`TRAIN_FROM` → `TRAIN_TO`) directly from your Spanner database using the SCD Type 2 logic inside `SpannerDataset`.

In [ ]:
print("Fetching training data from Spanner...")
snapshots = run_pipeline_local.step_ingest(train_args, snapshots_dir)
print(f"Ingested {len(snapshots)} training snapshots.")

In [ ]:
# ── Snapshot summary table ────────────────────────────────────────────────────
_hdr = (f"{'#':>3}  {'Timestamp':<26}  {'Routers':>9}  "
        f"{'Interfaces':>12}  {'BGPSessions':>13}  {'Edges':>6}")
print(f"\n{_hdr}")
print("-" * len(_hdr))
for _i, _s in enumerate(snapshots):
    _r   = [n for n in _s['nodes'] if n['type'] == 'router']
    _ifc = [n for n in _s['nodes'] if n['type'] == 'interface']
    _bgp = [n for n in _s['nodes'] if n['type'] == 'bgp_session']
    _r_up   = sum(1 for n in _r   if n.get('state') == 1.0)
    _i_up   = sum(1 for n in _ifc if n.get('state') == 1.0)
    _b_up   = sum(1 for n in _bgp if n.get('bgp_state') == 1.0)
    _ec = {}
    for _e in _s['edges']:
        _ec[_e['relation']] = _ec.get(_e['relation'], 0) + 1
    _etag = '  '.join(f"{k}={v}" for k, v in sorted(_ec.items()))
    print(
        f"{_i+1:>3}  {_s['timestamp']:<26}  "
        f"{_r_up}/{len(_r):<2} up  "
        f"{_i_up}/{len(_ifc):<4} up  "
        f"{_b_up}/{len(_bgp):<3} estab  "
        f"{len(_s['edges']):>6}  ({_etag})"
    )

# Show temporal features for last snapshot
if snapshots:
    _last = snapshots[-1]
    print("\n── Temporal features (last snapshot) ───────────────────────────")
    _ifc_nodes = [n for n in _last['nodes'] if n['type'] == 'interface']
    _bgp_nodes = [n for n in _last['nodes'] if n['type'] == 'bgp_session']
    _grads = sorted([abs(n.get('rx_err_gradient', 0.0)) for n in _ifc_nodes], reverse=True)
    print(f"  Interface rx_err_gradient   max={_grads[0] if _grads else 0:.6f}")
    _deltas = [n.get('prefix_count_delta', 0.0) for n in _bgp_nodes]
    print(f"  BGP prefix_count_delta      sessions with delta≠0: "
          f"{sum(1 for d in _deltas if d != 0)}/{len(_deltas)}")
    _uptimes = [n.get('session_uptime_norm', 0.0) for n in _bgp_nodes]
    print(f"  BGP session_uptime_norm     mean={sum(_uptimes)/len(_uptimes) if _uptimes else 0:.3f}")


## 1b. Ingest Test Data

Fetch the **held-out test snapshots** for the period immediately after training (`TEST_FROM` → `TEST_TO`). These are saved separately and used later to evaluate model performance on unseen data.

In [ ]:
print("Fetching test data from Spanner...")
test_snapshots = run_pipeline_local.step_ingest(test_args, test_snapshots_dir)
print(f"Ingested {len(test_snapshots)} test snapshots.")

In [ ]:
# ── Test snapshot summary ─────────────────────────────────────────────────────
_hdr = (f"{'#':>3}  {'Timestamp':<26}  {'Routers':>9}  "
        f"{'Interfaces':>12}  {'BGPSessions':>13}  {'Edges':>6}")
print(f"\n{_hdr}")
print("-" * len(_hdr))
for _i, _s in enumerate(test_snapshots):
    _r   = [n for n in _s['nodes'] if n['type'] == 'router']
    _ifc = [n for n in _s['nodes'] if n['type'] == 'interface']
    _bgp = [n for n in _s['nodes'] if n['type'] == 'bgp_session']
    _r_up   = sum(1 for n in _r   if n.get('state') == 1.0)
    _i_up   = sum(1 for n in _ifc if n.get('state') == 1.0)
    _b_up   = sum(1 for n in _bgp if n.get('bgp_state') == 1.0)
    _ec = {}
    for _e in _s['edges']:
        _ec[_e['relation']] = _ec.get(_e['relation'], 0) + 1
    _etag = '  '.join(f"{k}={v}" for k, v in sorted(_ec.items()))
    print(
        f"{_i+1:>3}  {_s['timestamp']:<26}  "
        f"{_r_up}/{len(_r):<2} up  "
        f"{_i_up}/{len(_ifc):<4} up  "
        f"{_b_up}/{len(_bgp):<3} estab  "
        f"{len(_s['edges']):>6}  ({_etag})"
    )


## 2. Train HetGNN

Pass the fetched snapshots into the training script. This step:
1. Fits the `StandardScaler` on numerical features.
2. Converts data into PyTorch `HeteroData` graphs.
3. Trains the autoencoder over the specified number of epochs.
4. Saves `model.pth`, `scalers.pkl`, and `model_stats.pth` into our local `notebook_artifacts` folder.

In [ ]:
print("Training HetGNN on downloaded snapshots...")
model, val_loss, gb = train_hetgnn.train_hetgnn_on_snapshots(
    snapshot_objects=snapshots,
    output_dir=str(output_dir),
    hidden_channels=64, # Matches production defaults
    num_layers=2,
    epochs_override=50  # Reduced for rapid feedback (default is 50)
)
print(f"Training completed. Validation Loss: {val_loss:.4f}")

## 3. Serve (Inference)

Test the production serving code.
In production, `serve.py` queries Google Cloud Storage to find the latest trained model (via `latest_run.json`). To test it locally, we temporarily patch those loader functions to point to the artifacts we just trained in our `notebook_artifacts` directory.

This execution will:
1. Reload the model and scalers we just saved.
2. Fetch one *new* real snapshot from Spanner.
3. Perform a forward pass through the model to compute node embeddings and MSE reconstruction error.
4. **Write the results** directly back to the Spanner `NodeEmbedding` table.
5. Print the anomaly scores.

In [ ]:
print("Setting up serving components with local artifacts instead of GCS...")

def mock_load_manifest():
    return {"run_id": "notebook_test_run"}

def mock_download_artefacts(manifest):
    return output_dir

async def test_serving():
    # We use unittest.mock.patch so serve.py doesn't try to download from GCS
    with patch("serve._load_manifest", side_effect=mock_load_manifest), \
         patch("serve._download_artefacts", side_effect=mock_download_artefacts), \
         patch("serve.SPANNER_INSTANCE", spanner_instance), \
         patch("serve.SPANNER_DATABASE", spanner_database), \
         patch("serve.GOOGLE_CLOUD_PROJECT", project_id):

        # Run the actual inference code
        results = await serve._run_inference()

        print("\n─── INFERENCE RESULTS ─────────────────────────────────────────")
        print(f"Snapshot timestamp : {results['snapshot_timestamp']}")
        print(f"Total anomalies    : {results['anomaly_count']}")

        # Per-type anomaly summary with per-type threshold
        thresholds = serve._cache.get("thresholds", serve._FALLBACK_THRESHOLDS)
        print("\n  Node Type        Nodes   Anomalies   Threshold")
        print("  " + "-"*52)
        for ntype, scores in results['anomaly_scores'].items():
            thresh = thresholds.get(ntype, 0.20)
            n_anom = sum(1 for s in scores if s > thresh)
            print(f"  {ntype:<18} {len(scores):>5}   {n_anom:>9}   MSE>{thresh:.2f}")

        # ── RCA decision tree for anomalous routers ──────────────────────────
        print("\n── RCA Decision Tree ───────────────────────────────────────────")
        r_scores  = results['anomaly_scores'].get('router', [])
        r_expls   = results.get('anomaly_explanations', {}).get('router', [])
        bgp_scores = results['anomaly_scores'].get('bgp_session', [])
        bgp_expls  = results.get('anomaly_explanations', {}).get('bgp_session', [])
        r_thresh  = thresholds.get('router', 0.15)
        b_thresh  = thresholds.get('bgp_session', 0.10)

        anom_routers = [(i, s) for i, s in enumerate(r_scores) if s > r_thresh]
        if not anom_routers:
            print("  No anomalous routers detected — network looks healthy.")
        else:
            for idx, score in anom_routers[:5]:   # Show first 5
                expl = r_expls[idx] if idx < len(r_expls) else {}
                top_f = expl.get('top_features', [])
                feat_str = ', '.join(f'{f["feature"]}' for f in top_f) if top_f else 'n/a'
                print(f"  ▶ Router[{idx}] MSE={score:.4f}  top features: {feat_str}")

                # Check if any BGP sessions on this router are also anomalous
                anom_bgp = [i for i, s in enumerate(bgp_scores) if s > b_thresh]
                if anom_bgp:
                    print(f"    ↳ {len(anom_bgp)} BGP session(s) also anomalous — possible RR failure")
                    for bi in anom_bgp[:3]:
                        b_expl = bgp_expls[bi] if bi < len(bgp_expls) else {}
                        b_top  = b_expl.get('top_features', [])
                        b_str  = ', '.join(f'{f["feature"]}' for f in b_top) if b_top else 'n/a'
                        print(f"       BGPSession[{bi}] MSE={bgp_scores[bi]:.4f}  top: {b_str}")
                else:
                    print(f"    ↳ BGP sessions healthy — likely interface or CPU issue")

# Await the async server inference method
await test_serving()
